<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">01 Lecture - The Performance Landscape</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Lecture - The Performance Landscape
#### Duration: 20 minutes

## Overview

This opening lecture builds the vocabulary that the rest of the course will use.

We start with what *fast* and *cheap* mean, because every optimization is judged against them. 

Then we look at how the engine runs your query, and what *partition*, *task*, and *shuffle* mean at a high level. 

With that in place, the reasons why a distributed query is slow become clear: it reads too much data, or it moves too much data across the network. Those two costs point at the levers we control, **Reading Less** and **Shuffling Less**.

Then we introduce what the platform now does for you. Predictive Optimization, Liquid Clustering, Adaptive Query Execution, Photon and serverless each automate those costs. 

We close with the workflow this course follows: let the defaults work, verify what the platform did in the Query Profile, then tune only what is left.

> Every capability named here gets its own hands-on demo later.

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Define the metrics** this course optimizes against: latency, throughput, and cost measured as DBUs per run.
2. **Describe how Spark runs a query**: partitions and tasks, lazy evaluation, and the difference between narrow and wide transformations.
3. **Explain why distributed queries are slow** in engine-neutral terms: reading too much data (I/O) and moving too much data (shuffle), plus skew, spill, and exploding joins.
4. **Name the two levers you control** (Read Less, Shuffle Less) and **what the platform now automates** (Predictive Optimization, Liquid Clustering, AQE and PQE, Photon, serverless), mapping each automation back to a lever.
5. **Apply the platform-first workflow**: defaults first, verify in the Query Profile, then tune only what remains.


## A. What "Fast" and "Cheap" Mean

Here are the three principles that we judge every change by:

- **Latency**: how long a single run takes, the wall-clock time a user waits on.
- **Throughput**: how much work finishes per unit of time, which is what matters for pipelines and concurrency.
- **Cost**: the platform resources a run consumes, measured here as **DBUs per run** and watched as a trend.
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.tc { text-align: center; }
.ds-stat { font-size: 26pt; font-weight: 800; color: var(--accent); line-height: 1.1; }
.ds-stat-label { font-size: 14pt; color: #5E7077; margin-top: 6px; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; margin-top: 20px; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
</style>

<div class="ds-row">
  <div class="ds-card blue tc"><div class="ds-stat">Duration</div><div class="ds-stat-label">Speed: elapsed time per run</div></div>
  <div class="ds-card coral tc"><div class="ds-stat">DBUs/Run</div><div class="ds-stat-label">Cost: platform resource units per execution</div></div>
  <div class="ds-card amber tc"><div class="ds-stat">Trend</div><div class="ds-stat-label">Direction: is cost moving up or down over time?</div></div>
</div>

<div class="ds-callout" style="--accent:#FF5F46; --tint:#FFF6F4;">
  <div class="ds-eyebrow">Key Point</div>
  A query that's <strong>10%</strong> faster but <strong>costs 2x more</strong> is not a win. <br></br>Track both metrics, together, on every run.
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Cost is a first-class metric
- Speed alone hides regressions. Bigger compute can make almost anything faster while quietly raising the bill.
- The unit of this course is DBUs per run, tracked over time. You attribute it per job, warehouse, and user, and you trend it, rather than discovering it at month-end.
- Later parts dollarize DBUs by joining `system_vw.billing.usage` to `system_vw.billing.list_prices`.
- Attribution needs tags and serverless usage policies. Untagged serverless spend is unattributable by design, so tagging is part of performance work.

</details>

## B. How the Engine Runs Your Query

Every performance idea in this course is based on how Spark actually executes your code. 

Here are a few terms that are worth understanding.

- **Partition**: one chunk of your data. Spark splits a table into many partitions so it can work on them independently. This is a *runtime* concept. Later modules use the word *partitioning* for a table layout on disk, which is a different concept and we'll cover that later.
- **Task**: the work of processing one partition on one CPU core. Every task in a step runs the same operation on a different partition, at the same time.
- **Parallelism**: how many tasks can run at once, which is the total number of cores. More partitions than cores means tasks run in waves, fewer partitions than cores means some cores sit idle.
- **Lazy evaluation**: transformations such as `filter`, `join`, and `groupBy` don't run as you write them. They build up a plan. Nothing executes until an **action** (`count`, `show`, `collect`, `write`) forces it, which is why a Query Profile only exists once an action has run.

The diagrams below show this in three steps: how work is spread across the cluster, then the single most important distinction for performance (narrow versus wide transformations), then how a full query becomes stages and tasks.

### B1. How Spark Distributes Work
<div class="g1">
<style>
.g1 { font-family: sans-serif; max-width: 1060px; margin: 0 auto; color: #0b2026; }
.g1 h4 { margin: 0 0 4px; font-size: 18pt; font-weight: 700; color: #1B5162; }
.g1 p.sub { margin: 0 0 16px; font-size: 14pt; color: #618794; line-height: 1.5; }
.g1 svg { display: block; width: 100%; height: auto; margin-bottom: 18px; }
.g1 text { font-family: sans-serif; }
.g1 .note { font-size: 14pt; color: #0b2026; border-left: 4px solid #4299E0; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; line-height: 1.55; margin-bottom: 16px; }
.g1 .note strong { color: #1B5162; }
.g1 .amber-box { background: rgba(255,171,0,0.12); border: 2px solid #FFAB00; border-radius: 10px; padding: 18px 22px; }
.g1 .amber-box ul { margin: 8px 0 0; padding-left: 22px; font-size: 14pt; line-height: 1.6; color: #0b2026; }
.g1 .amber-box li { margin-bottom: 8px; }
.g1 .amber-box li:last-child { margin-bottom: 0; }
</style>

<h4>The Execution Hierarchy</h4>
<p class="sub">Your code submits work to a driver, which distributes it across executors. Each executor owns CPU cores and each core runs one task (one partition) at a time.</p>

<svg viewBox="0 0 900 416" role="img" aria-label="Diagram: Spark execution hierarchy from notebook code to tasks on executor cores">
  <defs>
    <marker id="g1a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path></marker>
  </defs>
  <!-- YOUR CODE -->
  <rect x="10" y="140" width="160" height="70" rx="10" fill="#0b2026"></rect>
  <text x="90" y="170" text-anchor="middle" font-size="13" font-weight="700" fill="#FFFFFF">YOUR NOTEBOOK</text>
  <text x="90" y="190" text-anchor="middle" font-size="11" fill="#90A5B1">Python / SQL</text>
  <!-- Arrow to Driver -->
  <line x1="170" y1="175" x2="210" y2="175" stroke="#618794" stroke-width="2" marker-end="url(#g1a)"></line>
  <!-- DRIVER -->
  <rect x="210" y="130" width="160" height="90" rx="10" fill="#1B5162"></rect>
  <text x="290" y="158" text-anchor="middle" font-size="13" font-weight="700" fill="#FFFFFF">DRIVER</text>
  <text x="290" y="178" text-anchor="middle" font-size="11" fill="#C4CCD6">SparkSession</text>
  <text x="290" y="198" text-anchor="middle" font-size="10" fill="#C4CCD6">Plans + coordinates</text>
  <!-- Arrow to Cluster -->
  <line x1="370" y1="175" x2="420" y2="175" stroke="#618794" stroke-width="2" marker-end="url(#g1a)"></line>
  <text x="395" y="165" text-anchor="middle" font-size="10" fill="#618794">distributes</text>
  <!-- CLUSTER BOUNDARY -->
  <rect x="420" y="10" width="470" height="380" rx="12" fill="rgba(248,249,252,0.5)" stroke="#618794" stroke-width="2" stroke-dasharray="8 4"></rect>
  <text x="655" y="32" text-anchor="middle" font-size="11" font-weight="700" letter-spacing="0.5" fill="#618794">MANAGED BY DATABRICKS ON SERVERLESS</text>
  <!-- EXECUTOR 1 -->
  <rect x="440" y="48" width="210" height="320" rx="8" fill="#FFFFFF" stroke="#00A972" stroke-width="2"></rect>
  <rect x="440" y="48" width="210" height="34" rx="8" fill="#00A972"></rect>
  <rect x="440" y="72" width="210" height="10" fill="#00A972"></rect>
  <text x="545" y="70" text-anchor="middle" font-size="12" font-weight="700" fill="#FFFFFF">EXECUTOR 1</text>
  <!-- Photon badge -->
  <rect x="490" y="92" width="110" height="20" rx="4" fill="#98102A"></rect>
  <text x="545" y="106" text-anchor="middle" font-size="10" font-weight="700" fill="#FFFFFF">PHOTON (C++)</text>
  <!-- Cores label -->
  <text x="545" y="132" text-anchor="middle" font-size="11" font-weight="600" fill="#0b2026">4 Cores</text>
  <!-- Core boxes -->
  <rect x="455" y="142" width="90" height="55" rx="5" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.4"></rect>
  <text x="500" y="158" text-anchor="middle" font-size="10" font-weight="700" fill="#4299E0">Core 1</text>
  <rect x="465" y="164" width="70" height="24" rx="4" fill="rgba(66,153,224,0.12)" stroke="#4299E0"></rect>
  <text x="500" y="181" text-anchor="middle" font-size="11" font-weight="700" fill="#0b2026">Task</text>
  <rect x="555" y="142" width="90" height="55" rx="5" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.4"></rect>
  <text x="600" y="158" text-anchor="middle" font-size="10" font-weight="700" fill="#4299E0">Core 2</text>
  <rect x="565" y="164" width="70" height="24" rx="4" fill="rgba(66,153,224,0.12)" stroke="#4299E0"></rect>
  <text x="600" y="181" text-anchor="middle" font-size="11" font-weight="700" fill="#0b2026">Task</text>
  <rect x="455" y="205" width="90" height="55" rx="5" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.4"></rect>
  <text x="500" y="221" text-anchor="middle" font-size="10" font-weight="700" fill="#4299E0">Core 3</text>
  <rect x="465" y="227" width="70" height="24" rx="4" fill="rgba(66,153,224,0.12)" stroke="#4299E0"></rect>
  <text x="500" y="244" text-anchor="middle" font-size="11" font-weight="700" fill="#0b2026">Task</text>
  <rect x="555" y="205" width="90" height="55" rx="5" fill="#FFFFFF" stroke="#4299E0" stroke-width="1.4"></rect>
  <text x="600" y="221" text-anchor="middle" font-size="10" font-weight="700" fill="#4299E0">Core 4</text>
  <rect x="565" y="227" width="70" height="24" rx="4" fill="rgba(66,153,224,0.12)" stroke="#4299E0"></rect>
  <text x="600" y="244" text-anchor="middle" font-size="11" font-weight="700" fill="#0b2026">Task</text>
  <!-- Partition note -->
  <text x="545" y="290" text-anchor="middle" font-size="10" fill="#618794">1 task = 1 partition</text>
  <text x="545" y="306" text-anchor="middle" font-size="10" fill="#618794">of your data</text>
  <!-- EXECUTOR N (abbreviated) -->
  <rect x="670" y="48" width="200" height="320" rx="8" fill="#F8F9FC" stroke="#00A972" stroke-width="1.5" stroke-dasharray="6 3"></rect>
  <rect x="670" y="48" width="200" height="34" rx="8" fill="#00A972" opacity="0.5"></rect>
  <rect x="670" y="72" width="200" height="10" fill="#00A972" opacity="0.5"></rect>
  <text x="770" y="70" text-anchor="middle" font-size="12" font-weight="700" fill="#FFFFFF">EXECUTORS 2...N</text>
  <text x="770" y="110" text-anchor="middle" font-size="11" fill="#618794">Same structure</text>
  <!-- Gray placeholder cores -->
  <rect x="700" y="130" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <rect x="750" y="130" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <rect x="800" y="130" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <rect x="700" y="180" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <rect x="750" y="180" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <rect x="800" y="180" width="40" height="40" rx="4" fill="#EEEDE9"></rect>
  <!-- Parallelism note -->
  <text x="770" y="260" text-anchor="middle" font-size="11" font-weight="600" fill="#1B5162">Total cores =</text>
  <text x="770" y="278" text-anchor="middle" font-size="11" font-weight="600" fill="#1B5162">max parallelism</text>
  <!-- Bottom labels -->
  <text x="90" y="406" text-anchor="middle" font-size="10" fill="#618794">You write this</text>
  <text x="290" y="406" text-anchor="middle" font-size="10" fill="#618794">Plans your query</text>
  <text x="655" y="406" text-anchor="middle" font-size="10" fill="#618794">Processes your data in parallel</text>
</svg>

<div class="note">
  <strong>On serverless</strong> Databricks manages the entire cluster (executors, cores, scaling). But the concepts above still govern performance: tasks, partitions and cores determine parallelism. They appear directly in the Query Profile you'll read in the next lesson.
</div>

<div class="amber-box">
  <div style="font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 8px;">Key relationships</div>
  <ul>
    <li>Each executor owns a set of <strong>CPU cores</strong>. On serverless with Photon, the data processing engine is native C++ (not JVM), which eliminates garbage collection pauses and uses SIMD vectorized execution.</li>
    <li>Each core runs <strong>one task at a time</strong>. A task processes one partition of data.</li>
    <li><strong>Total cores across the cluster = maximum parallelism.</strong> If you have fewer partitions than cores, some cores sit idle. If you have far more, tasks run in waves.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

### The hierarchy

- **Your notebook.** You write transformations (filter, join, groupBy). These are lazy. They build a plan but do not run until an action triggers execution.
- **Driver (SparkSession).** The coordinator. It receives your plan, optimizes it through Catalyst, and distributes the resulting tasks to executors. On serverless with Spark Connect, the driver runs remotely. You interact with it through a thin client embedded in the notebook.
- **Executors.** Processes that do the actual data work. Each executor owns CPU cores and memory. On Databricks, Photon (a native C++ engine) handles scans, joins, aggregations, and writes. The JVM still coordinates, but data processing happens in Photon's columnar batch engine using SIMD instructions.
- **Cores.** Each executor has multiple cores. Each core can run one task at a time.
- **Tasks.** The smallest unit of work. One task processes one partition. All tasks in a stage do the same operation on different partitions, in parallel.

### Photon on serverless

- Photon is always on for serverless compute and SQL warehouses. You cannot disable it. There is no toggle, no config, no opt-out. The disable checkbox ("Use Photon Acceleration") exists only for classic all-purpose and jobs compute.
- It processes data in batches of thousands of rows at a time (columnar), using SIMD instructions that evaluate multiple values per CPU cycle.
- It eliminates JVM garbage collection pauses, JIT warm-up delays, and memory overhead.
- For unsupported operations (Python UDFs, RDD APIs, stateful streaming), Photon falls back to the Spark JVM runtime transparently.

### Why this matters for tuning

- The total number of cores is your ceiling on parallelism. On serverless, Databricks sizes this for you, but the partition-to-core ratio still determines whether you leave capacity idle or create too many small tasks.
- The Query Profile shows these concepts directly: each operator processes tasks in parallel, and the slowest task in a stage determines stage duration.

</details>


### B2. Narrow vs Wide Transformations
<div class="nw">
<style>
.nw { font-family: sans-serif; max-width: 1060px; margin: 0 auto; color: #0b2026; }
.nw * { box-sizing: border-box; }
.nw h4 { margin: 0 0 4px; font-size: 18pt; font-weight: 700; color: #1B5162; }
.nw p.sub { margin: 0 0 16px; font-size: 14pt; color: #618794; line-height: 1.5; }
.nw svg { display: block; width: 100%; height: auto; margin-bottom: 16px; }
.nw text { font-family: sans-serif; }
.nw .note { font-size: 14pt; color: #0b2026; border-left: 4px solid #98102A; background: rgba(152,16,42,0.06); padding: 14px 18px; border-radius: 4px; line-height: 1.55; }
.nw .note strong { color: #98102A; }
.nw code { font-size: 13pt; background: rgba(27,49,57,0.07); padding: 1px 5px; border-radius: 4px; }
</style>

<h4>Why <code>filter</code> is cheap and <code>groupBy</code> is not</h4>
<p class="sub">A narrow transformation processes each partition on its own, with nothing crossing the network. A wide transformation must move rows so that matching keys land together. That redistribution is the shuffle and it's the cost that "Shuffle Less" exists to reduce.</p>

<svg viewBox="0 0 900 300" role="img" aria-label="Narrow transformations process each partition independently with no network movement. Wide transformations redistribute rows across the network so matching keys land together, which is the shuffle.">
  <defs>
    <marker id="nwa" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path></marker>
  </defs>
  <text x="20" y="24" font-size="15" font-weight="700" fill="#00A972">Narrow: cheap</text>
  <text x="20" y="45" font-size="12" fill="#618794">filter, select, withColumn</text>
  <rect x="20" y="62" width="70" height="44" rx="6" fill="rgba(0,169,114,0.10)" stroke="#00A972"></rect>
  <rect x="20" y="116" width="70" height="44" rx="6" fill="rgba(0,169,114,0.10)" stroke="#00A972"></rect>
  <rect x="20" y="170" width="70" height="44" rx="6" fill="rgba(0,169,114,0.10)" stroke="#00A972"></rect>
  <text x="55" y="232" text-anchor="middle" font-size="11" fill="#618794">partitions in</text>
  <line x1="92" y1="84" x2="146" y2="84" stroke="#618794" stroke-width="1.6" marker-end="url(#nwa)"></line>
  <line x1="92" y1="138" x2="146" y2="138" stroke="#618794" stroke-width="1.6" marker-end="url(#nwa)"></line>
  <line x1="92" y1="192" x2="146" y2="192" stroke="#618794" stroke-width="1.6" marker-end="url(#nwa)"></line>
  <rect x="150" y="62" width="70" height="44" rx="6" fill="rgba(0,169,114,0.24)" stroke="#00A972"></rect>
  <rect x="150" y="116" width="70" height="44" rx="6" fill="rgba(0,169,114,0.24)" stroke="#00A972"></rect>
  <rect x="150" y="170" width="70" height="44" rx="6" fill="rgba(0,169,114,0.24)" stroke="#00A972"></rect>
  <text x="185" y="232" text-anchor="middle" font-size="11" fill="#618794">partitions out</text>
  <text x="120" y="258" text-anchor="middle" font-size="12" font-weight="700" fill="#00A972">no network: each partition works alone</text>
  <line x1="410" y1="14" x2="410" y2="270" stroke="#618794" stroke-width="1" stroke-dasharray="3 5" opacity="0.5"></line>
  <text x="450" y="24" font-size="15" font-weight="700" fill="#98102A">Wide: expensive</text>
  <text x="450" y="45" font-size="12" fill="#618794">groupBy, join, distinct, orderBy</text>
  <rect x="450" y="62" width="84" height="44" rx="6" fill="#F8F9FC" stroke="#618794"></rect>
  <circle cx="470" cy="84" r="6" fill="#4299E0"></circle><circle cx="492" cy="84" r="6" fill="#FFAB00"></circle><circle cx="514" cy="84" r="6" fill="#98102A"></circle>
  <rect x="450" y="116" width="84" height="44" rx="6" fill="#F8F9FC" stroke="#618794"></rect>
  <circle cx="470" cy="138" r="6" fill="#98102A"></circle><circle cx="492" cy="138" r="6" fill="#4299E0"></circle><circle cx="514" cy="138" r="6" fill="#FFAB00"></circle>
  <rect x="450" y="170" width="84" height="44" rx="6" fill="#F8F9FC" stroke="#618794"></rect>
  <circle cx="470" cy="192" r="6" fill="#FFAB00"></circle><circle cx="492" cy="192" r="6" fill="#98102A"></circle><circle cx="514" cy="192" r="6" fill="#4299E0"></circle>
  <text x="492" y="232" text-anchor="middle" font-size="11" fill="#618794">mixed keys</text>
  <rect x="558" y="62" width="46" height="152" rx="8" fill="rgba(152,16,42,0.10)" stroke="#98102A" stroke-width="1.3" stroke-dasharray="4 3"></rect>
  <text x="581" y="138" text-anchor="middle" font-size="11" font-weight="700" fill="#98102A" transform="rotate(-90 581 138)">SHUFFLE</text>
  <line x1="534" y1="84" x2="558" y2="84" stroke="#618794" stroke-width="1.2" opacity="0.6"></line>
  <line x1="534" y1="138" x2="558" y2="138" stroke="#618794" stroke-width="1.2" opacity="0.6"></line>
  <line x1="534" y1="192" x2="558" y2="192" stroke="#618794" stroke-width="1.2" opacity="0.6"></line>
  <line x1="604" y1="116" x2="640" y2="86" stroke="#4299E0" stroke-width="1.8" marker-end="url(#nwa)"></line>
  <line x1="604" y1="138" x2="640" y2="140" stroke="#FFAB00" stroke-width="1.8" marker-end="url(#nwa)"></line>
  <line x1="604" y1="160" x2="640" y2="194" stroke="#98102A" stroke-width="1.8" marker-end="url(#nwa)"></line>
  <rect x="642" y="64" width="84" height="44" rx="6" fill="rgba(66,153,224,0.14)" stroke="#4299E0"></rect>
  <circle cx="663" cy="86" r="6" fill="#4299E0"></circle><circle cx="685" cy="86" r="6" fill="#4299E0"></circle><circle cx="707" cy="86" r="6" fill="#4299E0"></circle>
  <rect x="642" y="118" width="84" height="44" rx="6" fill="rgba(255,171,0,0.16)" stroke="#FFAB00"></rect>
  <circle cx="663" cy="140" r="6" fill="#FFAB00"></circle><circle cx="685" cy="140" r="6" fill="#FFAB00"></circle><circle cx="707" cy="140" r="6" fill="#FFAB00"></circle>
  <rect x="642" y="172" width="84" height="44" rx="6" fill="rgba(152,16,42,0.14)" stroke="#98102A"></rect>
  <circle cx="663" cy="194" r="6" fill="#98102A"></circle><circle cx="685" cy="194" r="6" fill="#98102A"></circle><circle cx="707" cy="194" r="6" fill="#98102A"></circle>
  <text x="684" y="232" text-anchor="middle" font-size="11" fill="#618794">grouped by key</text>
  <text x="584" y="258" text-anchor="middle" font-size="12" font-weight="700" fill="#98102A">rows cross the network so like keys meet</text>
</svg>

<div class="note"><strong>The shuffle</strong> is the single most expensive thing most queries do. 
<br />
Any operation that groups or matches rows by key (<code>groupBy</code>, <code>join</code>, <code>distinct</code>, <code>orderBy</code>) triggers one. Every later demo about skew, spill and joins is really about making shuffles smaller or avoiding them.</div>
</div>

### B3. From Code to Results
<div class="g2">
<style>
.g2 { font-family: sans-serif; max-width: 1060px; margin: 0 auto; color: #0b2026; }
.g2 * { box-sizing: border-box; }
.g2 h4 { margin: 0 0 4px; font-size: 18pt; font-weight: 700; color: #1B5162; }
.g2 p.sub { margin: 0 0 18px; font-size: 14pt; color: #618794; line-height: 1.5; }
.g2 .step-list { display: flex; flex-direction: column; gap: 0; margin-bottom: 22px; }
.g2 .step { background: #F8F9FC; border: 1.5px solid #618794; border-left: 6px solid #4299E0; border-radius: 8px; padding: 14px 18px; }
.g2 .step-head { font-size: 14pt; font-weight: 700; color: #0b2026; }
.g2 .step-head span { font-weight: 400; color: #618794; }
.g2 .step-body { font-size: 14pt; color: #5E7077; margin-top: 4px; line-height: 1.5; }
.g2 .arr { text-align: center; color: #618794; font-size: 20pt; line-height: 1; padding: 2px 0; }
.g2 .aqe-row { display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 22px; }
.g2 .aqe-card { flex: 1; min-width: 200px; background: rgba(255,171,0,0.10); border: 2px solid #FFAB00; border-radius: 8px; padding: 14px 16px; }
.g2 .aqe-card-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 4px; }
.g2 .aqe-card-text { font-size: 14pt; color: #5E7077; line-height: 1.5; }
.g2 svg { display: block; width: 100%; height: auto; margin-bottom: 18px; }
.g2 text { font-family: sans-serif; }
</style>

<h4>From Code to Tasks: How Spark Runs a Query</h4>
<p class="sub">Your transformations are lazy. An action triggers execution: Spark plans the query, then runs it as stages and tasks across the cluster.</p>

<!-- ===== 5-STEP FLOW ===== -->
<div class="step-list">
  <div class="step"><div class="step-head">1. Write Transformations <span>- lazy, nothing runs yet</span></div><div class="step-body">You describe what you want: filter, join, groupBy, select. Spark builds an unresolved logical plan.</div></div>
  <div class="arr">&darr;</div>
  <div class="step"><div class="step-head">2. Action <span>- count, show, collect, write</span></div><div class="step-body">An action triggers execution. Spark now must produce a result.</div></div>
  <div class="arr">&darr;</div>
  <div class="step"><div class="step-head">3. Catalyst Optimizes <span>- logical plan to physical plan</span></div><div class="step-body">Catalyst (the optimizer) resolves references, pushes filters down, chooses join strategies, and produces a physical plan.</div></div>
  <div class="arr">&darr;</div>
  <div class="step"><div class="step-head">4. Job &rarr; Stages <span>- stages split at shuffles</span></div><div class="step-body">A job is split into stages at every shuffle boundary (Exchange operator). Within a stage, tasks process partitions in parallel without network I/O.</div></div>
  <div class="arr">&darr;</div>
  <div class="step"><div class="step-head">5. Stages &rarr; Tasks <span>- one task per partition per core</span></div><div class="step-body">Each stage runs tasks in parallel across executor cores. Task count = number of partitions in that stage.</div></div>
</div>

<!-- ===== AQE + PQE BRIDGE ===== -->
<div style="font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px;">Between steps 3 and 5, Adaptive Query Execution (AQE) adapts the plan</div>
<div class="aqe-row">
  <div class="aqe-card">
    <div class="aqe-card-title">AQE: Coalesce Partitions</div>
    <div class="aqe-card-text">After each shuffle, AQE measures actual partition sizes and merges small ones together. Eliminates thousands of tiny tasks.</div>
  </div>
  <div class="aqe-card">
    <div class="aqe-card-title">AQE: Handle Skew</div>
    <div class="aqe-card-text">Detects partitions much larger than their siblings and splits them so no single task becomes a straggler.</div>
  </div>
  <div class="aqe-card">
    <div class="aqe-card-title">AQE + PQE: Switch Strategy</div>
    <div class="aqe-card-text">If a table is small enough at runtime, AQE converts to broadcast join. PQE (Predictive Query Execution) adds further replanning based on live signals like spill pressure.</div>
  </div>
</div>

<!-- ===== SVG: COLORED-KEY SHUFFLE VISUALIZATION ===== -->
<div style="font-size: 14pt; font-weight: 700; color: #1B5162; margin-bottom: 8px;">What happens at each stage boundary (shuffle)</div>
<p class="sub">Data is redistributed by key so that all rows with the same key land on the same executor. This is necessary for joins and aggregations.</p>

<svg viewBox="0 0 900 300" role="img" aria-label="Diagram: Colored partitions redistributed across shuffle boundaries between stages">
  <defs>
    <marker id="g2a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path></marker>
  </defs>
  <!-- STAGE 1 LABEL -->
  <text x="120" y="18" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162">STAGE 1: Scan + Filter</text>
  <!-- Stage 1 partitions (mixed colors = mixed keys) -->
  <text x="25" y="46" font-size="10" fill="#618794">Partition 1</text>
  <rect x="20" y="52" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#618794" stroke-width="1.5"></rect>
  <rect x="28" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="56" y="59" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="84" y="59" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="112" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="140" y="59" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="168" y="59" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <text x="25" y="112" font-size="10" fill="#618794">Partition 2</text>
  <rect x="20" y="118" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#618794" stroke-width="1.5"></rect>
  <rect x="28" y="125" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="56" y="125" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="84" y="125" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="112" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="140" y="125" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="168" y="125" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <text x="25" y="178" font-size="10" fill="#618794">Partition 3</text>
  <rect x="20" y="184" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#618794" stroke-width="1.5"></rect>
  <rect x="28" y="191" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="56" y="191" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="84" y="191" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="112" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="140" y="191" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="168" y="191" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <!-- SHUFFLE BOX -->
  <rect x="245" y="95" width="100" height="86" rx="6" fill="rgba(255,171,0,0.08)" stroke="#FFAB00" stroke-width="1.5" stroke-dasharray="4 2"></rect>
  <text x="295" y="126" text-anchor="middle" font-size="11" font-weight="700" fill="#FFAB00">SHUFFLE</text>
  <text x="295" y="145" text-anchor="middle" font-size="10" fill="#618794">Exchange</text>
  <text x="295" y="160" text-anchor="middle" font-size="10" fill="#618794">operator</text>
  <line x1="220" y1="138" x2="245" y2="138" stroke="#618794" stroke-width="2" marker-end="url(#g2a)"></line>
  <line x1="345" y1="138" x2="370" y2="138" stroke="#618794" stroke-width="2" marker-end="url(#g2a)"></line>
  <!-- STAGE 2 LABEL -->
  <text x="480" y="18" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162">STAGE 2: Grouped by Key</text>
  <!-- Stage 2 partitions (same-color grouped) -->
  <text x="385" y="46" font-size="10" fill="#4299E0" font-weight="600">Blue keys</text>
  <rect x="380" y="52" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#4299E0" stroke-width="2"></rect>
  <rect x="388" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="416" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="444" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="472" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="500" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <rect x="528" y="59" width="24" height="26" rx="3" fill="#4299E0"></rect>
  <text x="385" y="112" font-size="10" fill="#FF5F46" font-weight="600">Coral keys</text>
  <rect x="380" y="118" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#FF5F46" stroke-width="2"></rect>
  <rect x="388" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="416" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="444" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="472" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="500" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <rect x="528" y="125" width="24" height="26" rx="3" fill="#FF5F46"></rect>
  <text x="385" y="178" font-size="10" fill="#00A972" font-weight="600">Green keys</text>
  <rect x="380" y="184" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#00A972" stroke-width="2"></rect>
  <rect x="388" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="416" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="444" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="472" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <rect x="500" y="191" width="24" height="26" rx="3" fill="#00A972"></rect>
  <!-- Amber partition -->
  <text x="385" y="244" font-size="10" fill="#FFAB00" font-weight="600">Amber keys</text>
  <rect x="380" y="250" width="200" height="40" rx="6" fill="#F8F9FC" stroke="#FFAB00" stroke-width="2"></rect>
  <rect x="388" y="257" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="416" y="257" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="444" y="257" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <rect x="472" y="257" width="24" height="26" rx="3" fill="#FFAB00"></rect>
  <!-- Right side annotation -->
  <text x="610" y="65" font-size="11" fill="#1B5162" font-weight="600">After shuffle:</text>
  <text x="610" y="86" font-size="11" fill="#618794">All rows with the same</text>
  <text x="610" y="103" font-size="11" fill="#618794">join/group key are now</text>
  <text x="610" y="120" font-size="11" fill="#618794">on the same executor.</text>
  <text x="610" y="152" font-size="11" fill="#618794">Joins and aggregations</text>
  <text x="610" y="169" font-size="11" fill="#618794">can now run locally</text>
  <text x="610" y="186" font-size="11" fill="#618794">without further network.</text>
  <!-- Skew indicator -->
  <text x="610" y="224" font-size="11" font-weight="700" fill="#98102A">Skew = one partition</text>
  <text x="610" y="241" font-size="11" font-weight="700" fill="#98102A">much larger than others</text>
  <text x="610" y="262" font-size="11" fill="#618794">(Demo 07 covers fixes)</text>
</svg>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

### The execution path in full

- **Lazy transformations** build a directed acyclic graph (DAG) of logical operations. Nothing touches data until an action.
- **Catalyst** (the optimizer) rewrites the logical plan: predicate pushdown, column pruning, join reordering. Then it picks physical operators (broadcast join vs shuffle hash join, hash aggregate vs sort aggregate).
- **A job** is created per action. The physical plan is split into stages at shuffle boundaries. A shuffle happens whenever data must be redistributed by a new key (joins, groupBys, repartition).
- **Within a stage**, all tasks do the same operation on different partitions in parallel. No network I/O within a stage.
- **At stage boundaries** (Exchange operators), data is redistributed across the cluster by the new key. This is the shuffle.

### AQE + PQE details

- **AQE (Adaptive Query Execution)** is default-on since Spark 3.2. It re-optimizes the plan between stages, after it has real statistics from the completed stage.
  - Coalesces small post-shuffle partitions (merges 200 default partitions into fewer, larger ones).
  - Detects skewed partitions and splits them for parallel processing.
  - Converts shuffle joins (shown as "Photon Shuffled Hash Join" in the Query Profile) to broadcast joins when one side turns out to be small.
  - Detects empty relations and short-circuits them.
- **PQE (Predictive Query Execution)** extends this on Databricks. It adds runtime replanning based on signals from the native execution engine: spill pressure, cache behavior, and intermediate sizes. PQE can change strategies mid-execution when Photon detects conditions that the static optimizer could not predict.

</details>


## C. Why Distributed Queries Are Slow
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.cw p.intro { font-size: 14pt; line-height: 1.55; margin: 0 0 18px; }
.cw .ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.cw .ds-row > * { flex: 1; min-width: 260px; }
.cw .ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.cw .blue{--accent:#4299E0;} .cw .coral{--accent:#FF5F46;}
.cw .ds-card-title { font-size: 18pt; font-weight: 700; margin: 0 0 4px; }
.cw .ds-card-sub { font-size: 13pt; font-weight: 600; color: #1B5162; margin: 0 0 10px; }
.cw .ds-card p { font-size: 14pt; color: #0b2026; line-height: 1.55; margin: 0; }
.cw .fail { margin-top: 20px; border: 2px solid #EEEDE9; border-radius: 8px; background: #F8F9FC; padding: 18px 22px; }
.cw .fail-head { font-size: 14pt; font-weight: 700; color: #1B5162; margin-bottom: 10px; }
.cw .fail ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.6; }
.cw .fail li { margin-bottom: 8px; } .cw .fail li:last-child { margin-bottom: 0; }
</style>

<p class="intro">This is true of any distributed engine, not only Databricks. A query is slow for one of two physical reasons, and often both.</p>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-title">Reading too much data</div>
    <div class="ds-card-sub">The I/O cost</div>
    <p>Scanning files and rows you do not need. You attack it with good data layout so filters can skip files, and by not opening thousands of tiny files. Hands-on in the Data Layout and Liquid Clustering demos.</p>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-title">Moving too much data</div>
    <div class="ds-card-sub">The network cost, usually the expensive one</div>
    <p>The shuffle you just met. Every wide operation redistributes rows across the network. You attack it by moving fewer rows and choosing the right join strategy. Hands-on in the Liquid Clustering and Shuffle demos.</p>
  </div>
</div>

<div class="fail">
  <div class="fail-head">Three ways these costs get worse</div>
  <ul>
    <li><strong>Skew</strong>: a few keys hold most of the rows, so a few tasks do most of the work while the rest sit idle.</li>
    <li><strong>Spill</strong>: a task runs out of memory during a shuffle and has to write to disk, which is far slower than memory.</li>
    <li><strong>Exploding joins</strong>: a join multiplies rows before an aggregation, so later steps carry far more data than the inputs suggested.</li>
  </ul>
  <div style="margin-top:10px;font-size:13pt;color:#5E7077;">All three get hands-on treatment in the Shuffle, Skew, Spill and Explosion demo.</div>
</div>
</div>


## D. The Two Levers We Control

**Read Less** aims to reduce the I/O cost.

**Shuffle Less** aims to reduce the network cost. 

For everything else, the platform now automates for you.

<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 14pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.ds-card-sub { font-size: 14pt; font-weight: 600; color: #1B5162; margin: 0 0 12px; line-height: 1.4; }
</style>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-title">Read Less</div>
    <div class="ds-card-sub">Touch fewer bytes and open fewer files on every scan.</div>
    <ul>
      <li>Good data layout and clustering so filters prune files</li>
      <li>Column and file statistics that feed data skipping</li>
      <li>Caching and avoiding the small-file problem</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-title">Shuffle Less</div>
    <div class="ds-card-sub">Move fewer rows across the network.</div>
    <ul>
      <li>Smarter joins and aggregations</li>
      <li>Skew and spill handled at runtime</li>
      <li>Avoiding shuffles you don't need</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The two levers you control
Every recommendation in this course maps back to reading less or shuffling less. When you hit a slow workload later, ask which of the two it belongs to first. The third piece, automation, is the platform's job rather than a knob you turn, and it gets its own section next.

### Read Less
- Layout, clustering, and statistics let the engine skip files and row groups that cannot match your filter.
- Small files defeat this. Thousands of tiny files add scan and listing overhead, which is why self-tuning managed tables matter.

### Shuffle Less
- Shuffle moves rows across the network to line up joins and aggregations. It is often the most expensive thing a query does.
- Skew (a few huge keys) and spill (data that overflows memory to disk) are shuffle problems. Adaptive Query Execution handles the common cases at runtime, so hand-tuning is rarely the first move.

</details>


## E. What the Platform Now Automates for You
<br />
<div class="bnr-wrap">
<style>
.bnr-wrap { font-family: sans-serif; max-width: 1100px; margin: 0 auto; display: flex; flex-direction: column; gap: 20px; }
.bnr-wrap * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">The platform does the heavy lifting now</div>
    <div class="ds-banner-tagline">In 2026, data layout, table maintenance and much of runtime tuning are automatic. Your job shifts from immediate tuning to verifying the platform did its work, then tuning only what is left.</div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

- We're already familiar with Jobs, pipelines, PySpark, and SQL. What changed is where the optimization work happens.
- A few years ago, performance meant hand-tuning Spark configs, sizing executors, and choosing partition columns by hand.
- Today the platform owns most of that. The skill is knowing what runs automatically, confirming it actually engaged, and spending your effort only on the part the platform cannot infer.
- Keep one framing in mind for the whole course: read less, shuffle less, automate the rest and treat cost the same way you treat speed.

</details>


Each capability below **automates one of the levers you just learned**.
<br></br>

<div class="rc-wrap">
<style>
.rc-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }
.rc-wrap * { box-sizing: border-box; }
.rc-hint { width: 100%; max-width: 1100px; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; max-width: 1100px; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 90px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 14pt; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; max-width: 1100px; }
.rc-detail-wrap.open { max-height: 700px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body li:last-child { margin-bottom: 0; }
</style>

<div class="rc-hint"><strong>Click each card</strong> to see what it does and the prerequisite it carries.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Predictive Optimization</span>
    <span class="rc-title">Automatic table maintenance on UC managed tables</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rc-label" style="color:#00A972;">Liquid Clustering</span>
    <span class="rc-title">Lays out data around your filters, not fixed partitions</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#FFAB00; border-top-color:#FFAB00;">
    <span class="rc-label" style="color:#0b2026;">AQE + PQE</span>
    <span class="rc-title">Adapts query plans at runtime when reality differs</span>
  </span>
</div>

<div class="rc-row" style="text-align:center;">
  <span class="rc-box" data-id="3" onclick="rcSelect(3)" style="--pc:#98102A; border-top-color:#98102A; margin-top:18px;">
    <span class="rc-label" style="color:#98102A;">Photon</span>
    <span class="rc-title">Accelerates scans, joins, aggregations and writes</span>
  </span><span class="rc-box" data-id="4" onclick="rcSelect(4)" style="--pc:#4299E0; border-top-color:#4299E0; margin-top:18px;">
    <span class="rc-label" style="color:#4299E0;">Serverless</span>
    <span class="rc-title">Owns the compute so you stop sizing clusters</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>

<div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:14px 18px;border-radius:4px;margin:8px 0 0;width:100%;max-width:1100px;font-family:sans-serif;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">Automatic doesn't mean unconditional. Most of these are gated on <strong>Unity Catalog managed tables</strong> and several depend on <strong>Predictive Optimization</strong>. That's why in this course we will always verify a capability engaged before crediting it.</div>
</div>
</div>

<script>
var REVEAL = [
  { title: "Predictive Optimization", color: "#4299E0", aws: "https://docs.databricks.com/aws/en/optimizations/predictive-optimization", azure: "https://learn.microsoft.com/en-us/azure/databricks/optimizations/predictive-optimization", gcp: "https://docs.databricks.com/gcp/en/optimizations/predictive-optimization", text: "<ul><li>Runs <code>OPTIMIZE</code>, <code>VACUUM</code>, and <code>ANALYZE</code> for <strong>Unity Catalog managed tables</strong>.</li><li>It is asynchronous. You verify the work in <code>system_vw.storage.predictive_optimization_operations_history</code>.</li><li>It is the maintenance layer that makes managed tables feel self tuning.</li></ul>" },
  { title: "Liquid Clustering", color: "#00A972", aws: "https://docs.databricks.com/aws/en/delta/clustering", azure: "https://learn.microsoft.com/en-us/azure/databricks/delta/clustering", gcp: "https://docs.databricks.com/gcp/en/delta/clustering", text: "<ul><li>Replaces the old partitioning versus ZORDER decision for many analytics tables.</li><li>You cluster on the columns people filter on, or use <code>CLUSTER BY AUTO</code> where supported.</li><li><code>CLUSTER BY AUTO</code> needs a <strong>UC managed Delta table</strong> plus <strong>Predictive Optimization</strong>.</li></ul>" },
  { title: "AQE + PQE", color: "#FFAB00", aws: "https://docs.databricks.com/aws/en/optimizations/aqe", azure: "https://learn.microsoft.com/en-us/azure/databricks/optimizations/aqe", gcp: "https://docs.databricks.com/gcp/en/optimizations/aqe", text: "<ul><li><strong>AQE</strong> coalesces shuffle partitions, handles skew, and can switch join strategy at runtime.</li><li><strong>PQE</strong> adds Databricks and Photon runtime replanning based on live signals such as spill.</li><li>The habit we retire is hard coding shuffle behavior before the query even starts.</li></ul>" },
  { title: "Photon", color: "#98102A", aws: "https://docs.databricks.com/aws/en/runtime/photon", azure: "https://learn.microsoft.com/en-us/azure/databricks/runtime/photon", gcp: "https://docs.databricks.com/gcp/en/runtime/photon", text: "<ul><li>Photon is Databricks' vectorized engine for scans, joins, aggregations, and writes.</li><li>It is always on for serverless and SQL warehouses.</li><li>It falls back where needed, such as Python UDF heavy paths or RDD code, so the fast path is automatic when possible.</li></ul>" },
  { title: "Serverless", color: "#4299E0", aws: "https://docs.databricks.com/aws/en/compute/serverless", azure: "https://learn.microsoft.com/en-us/azure/databricks/compute/serverless", gcp: "https://docs.databricks.com/gcp/en/compute/serverless", text: "<ul><li>Serverless is now the default compute model across notebooks, Jobs, DBSQL warehouses, and Lakeflow pipelines.</li><li>You stop choosing node types, autoscaling ranges, and autotermination settings for most workloads.</li><li>The optimization workflow shifts up a level. Start here, then verify what the platform did.</li></ul>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div><div style="display:flex;gap:10px;margin-top:12px;"><a href="' + d.aws + '" target="_blank" style="display:inline-block;background:#1B5162;color:white;font-size:14pt;font-weight:700;padding:10px 24px;border-radius:8px;text-decoration:none;">AWS &rarr;</a><a href="' + d.azure + '" target="_blank" style="display:inline-block;background:#1B5162;color:white;font-size:14pt;font-weight:700;padding:10px 24px;border-radius:8px;text-decoration:none;">Azure &rarr;</a><a href="' + d.gcp + '" target="_blank" style="display:inline-block;background:#1B5162;color:white;font-size:14pt;font-weight:700;padding:10px 24px;border-radius:8px;text-decoration:none;">GCP &rarr;</a></div>';
  wrap.classList.add('open');
}
</script>


## F. The Workflow vs Old Advice You May Find Online
<br />

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 14pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
.dd-head { font-size: 20pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
.ds-card code { font-size: 13pt; background: rgba(27,49,57,0.07); padding: 1px 5px; border-radius: 4px; }
.flow-grid { display: grid; gap: 0; align-items: stretch; }
.flow-step { background: #F9F7F4; border: 3px solid var(--accent, #4299E0); border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0; }
.flow-num { font-size: 26pt; font-weight: 800; color: var(--accent, #4299E0); line-height: 1; margin-bottom: 8px; }
.flow-label { font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35; }
.flow-arrow { display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1; }
.flow-step.dim { background: #EEEDE9; border-color: #C2C2C2; box-shadow: none; opacity: 0.7; }
.flow-step.dim .flow-num { color: #90A5B1; }
.flow-step.dim .flow-label { color: #5A6F77; }
.flow-arrow.dim { color: #C2C2C2; }
</style>

<div class="flow-grid" style="grid-template-columns: 1fr auto 1fr auto 1fr; margin-bottom: 24px;">
  <div class="flow-step" style="--accent:#4299E0;"><div class="flow-num">1</div><div class="flow-label">Platform First</div></div>
  <div class="flow-arrow">&rarr;</div>
  <div class="flow-step" style="--accent:#C2C2C2; background:#F9F7F4; border-color:#C2C2C2; box-shadow:none;"><div class="flow-num" style="color:#90A5B1;">2</div><div class="flow-label" style="color:#5A6F77;">Verify</div></div>
  <div class="flow-arrow dim">&rarr;</div>
  <div class="flow-step dim" style="--accent:#FF5F46;"><div class="flow-num">3</div><div class="flow-label">Tune</div></div>
</div>

<div class="ds-row">
  <div class="ds-card green">
    <div class="dd-head">&check; Do This</div>
    <ul>
      <li>Start on <strong>managed tables</strong> and <strong>serverless</strong> so the defaults apply automatically.</li>
      <li>Let Predictive Optimization, AQE and Photon do their work.</li>
      <li>Open the <strong>Query Profile</strong> to confirm what actually engaged: pruning, cache hits, Photon, skew handling.</li>
      <li>Reach for a manual tuning only if you find a gap.</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="dd-head">&times; Old Advice You May Find Online</div>
    <ul>
      <li>Blog posts that set <code>spark.sql.shuffle.partitions=200</code> or other values. AQE now sizes shuffle partitions from the real data at runtime.</li>
      <li>Answers that set specific executors sizes. Serverless and Photon manage resources for you.</li>
      <li>Tutorials that sprinkle <code>.cache()</code> and <code>.persist()</code>. The disk and result caches warm themselves.</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Platform first, then verify, then tune

- The order matters. If you start tuning before the platform has had a chance to work, you often fight defaults or fix something the platform already handled.
- Later modules show exactly which Query Profile metrics confirm pruning, a cache hit, Photon engagement and skew handling.

### Old advice you'll meet online

You do not need to have written any of this yourself. The point is to recognize it in a blog post or a Stack Overflow answer and know why it no longer applies.

- `spark.sql.shuffle.partitions` pinned to 200 was a classic default that rarely matched actual data. AQE sizes shuffle partitions from the real data at runtime, so a hard-coded value usually hurts more than it helps.
- Sizing executors and GC flags was necessary when you owned the cluster. On serverless you don't, and Photon changes the memory model.
- Manual `.cache()` and `.persist()` was common. The disk cache and the DBSQL result cache handle the common cases automatically. Also on serverless, `.cache()` raises an `AnalysisException` rather than doing nothing.
</details>


## Conclusion

In this lecture, you:

1. Defined the metrics this course optimizes against: **latency, throughput, and cost** as DBUs per run, and the rule that faster plus more expensive is not a win.
2. Saw **how the engine runs your code**: partitions and tasks, lazy evaluation, and narrow versus wide transformations, where the wide ones cause the shuffle.
3. Learned **why distributed queries are slow**: reading too much data and moving too much data, plus skew, spill, and exploding joins.
4. Named the **two levers you control**, Read Less and Shuffle Less, and met **what the platform now automates** (Predictive Optimization, Liquid Clustering, AQE and PQE, Photon, serverless), each mapped back to a lever.
5. Adopted the **platform-first workflow**: defaults first, verify in the Query Profile, then tune only what is left.

Next, in **Measure First: Query Profiles and Benchmarks**, you pick up the instrument the rest of the course depends on: reading a Query Profile, ranking Performance Insights, and benchmarking honestly with the cache turned off.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>